In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


In [2]:
# --- Download NLTK resources (only needs to be done once) ---
print("Downloading NLTK resources...")
nltk.download('stopwords')
nltk.download('wordnet')
print("Downloads complete.")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arsha\AppData\Roaming\nltk_data...


Downloads complete.


[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\arsha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:

# --- 1. Text Cleaning and Lemmatization ---
print("\nPreprocessing text data...")

# Initialize the lemmatizer and stopwords list
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


Preprocessing text data...


In [4]:
def preprocess_text(text):
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize (split into words)
    words = text.split()
    # Lemmatize and remove stopwords
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

In [11]:
import pandas as pd
df = pd.read_csv(r'C:\Users\arsha\Downloads\suppor-ticket-classifier\data\complaints.csv')

In [12]:
# Apply the preprocessing function to the 'Description' column
# This might take a minute or two to run on all 58k rows
df['Cleaned Description'] = df['Description'].apply(preprocess_text)

print("Text preprocessing complete.")
print("\nOriginal vs. Cleaned Description:")
display(df[['Description', 'Cleaned Description']].head())

Text preprocessing complete.

Original vs. Cleaned Description:


,Description,Cleaned Description
0,XXXX has claimed I owe them {$27.00} for XXXX ...,xxxx claimed owe xxxx year despite proof payme...
1,In XX/XX/XXXX my wages that I earned at my job...,xx xx xxxx wage earned job decreased almost ha...
2,I have an open and current mortgage with Chase...,open current mortgage chase bank xxxx chase re...
3,XXXX was submitted XX/XX/XXXX. At the time I s...,xxxx submitted xx xx xxxx time submitted compl...
4,Experian is reporting my OPEN and CURRENT Mort...,experian reporting open current mortgage loan ...


In [18]:
df.to_csv('complaints_cleaned.csv', index=False)

### Data Splitting

In [13]:
# --- 2. Data Splitting ---
# We split the data into training (80%) and testing (20%) sets.
X = df['Cleaned Description']
y = df['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# 'stratify=y' ensures the category distribution is the same in train and test sets

print("\nData splitting complete.")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")


Data splitting complete.
Training data shape: (46909,)
Testing data shape: (11728,)


### TF IDF vectorize

In [14]:
# --- 3. TF-IDF Vectorization ---
print("\nPerforming TF-IDF Vectorization...")
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limit to top 5000 features for efficiency

# Fit on training data and transform both train and test data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Vectorization complete.")
print(f"Shape of TF-IDF matrix for training data: {X_train_tfidf.shape}")
print(f"Shape of TF-IDF matrix for testing data: {X_test_tfidf.shape}")


Performing TF-IDF Vectorization...
Vectorization complete.
Shape of TF-IDF matrix for training data: (46909, 5000)
Shape of TF-IDF matrix for testing data: (11728, 5000)


In [17]:
import joblib
from scipy.sparse import save_npz
import os

# --- Create a directory to save the processed data ---
output_dir = 'processed_data'
os.makedirs(output_dir, exist_ok=True)
print(f"Created directory: '{output_dir}'")

# --- Export the processed data and the vectorizer ---

# Save the TF-IDF matrices
save_npz(os.path.join(output_dir, 'X_train_tfidf.npz'), X_train_tfidf)
save_npz(os.path.join(output_dir, 'X_test_tfidf.npz'), X_test_tfidf)
print("Saved TF-IDF matrices.")

# Save the labels (y_train and y_test)
joblib.dump(y_train, os.path.join(output_dir, 'y_train.joblib'))
joblib.dump(y_test, os.path.join(output_dir, 'y_test.joblib'))
print("Saved training and testing labels.")

# Save the TF-IDF vectorizer itself
joblib.dump(tfidf_vectorizer, os.path.join(output_dir, 'tfidf_vectorizer.joblib'))
print("Saved the TF-IDF vectorizer.")

print("\nAll files have been successfully saved!")

Created directory: 'processed_data'
Saved TF-IDF matrices.
Saved training and testing labels.
Saved the TF-IDF vectorizer.

All files have been successfully saved!


### Modeling Trial

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# --- 1. Train and Evaluate Logistic Regression ---
print("--- Training Logistic Regression Model ---")
# We increase max_iter to ensure the model converges
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred_log_reg = log_reg.predict(X_test_tfidf)

# Evaluate the model
accuracy_lr = accuracy_score(y_test, y_pred_log_reg)
print(f"\nLogistic Regression Accuracy: {accuracy_lr:.4f}")
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log_reg))


# --- 2. Train and Evaluate Multinomial Naive Bayes ---
print("\n--- Training Multinomial Naive Bayes Model ---")
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred_nb = nb_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print(f"\nNaive Bayes Accuracy: {accuracy_nb:.4f}")
print("Naive Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))

--- Training Logistic Regression Model ---

Logistic Regression Accuracy: 0.8938
Logistic Regression Classification Report:
                         precision    recall  f1-score   support

Bank account or service       0.87      0.82      0.84      1142
            Credit card       0.85      0.83      0.84      1586
       Credit reporting       0.89      0.87      0.88      2505
        Debt collection       0.88      0.91      0.90      3511
               Mortgage       0.95      0.96      0.95      2984

               accuracy                           0.89     11728
              macro avg       0.89      0.88      0.88     11728
           weighted avg       0.89      0.89      0.89     11728


--- Training Multinomial Naive Bayes Model ---

Naive Bayes Accuracy: 0.8586
Naive Bayes Classification Report:
                         precision    recall  f1-score   support

Bank account or service       0.91      0.73      0.81      1142
            Credit card       0.82      0.76